<a href="https://colab.research.google.com/github/sredward11/projeto_machine_learning/blob/feat%2Fmarco1-eda-baseline/01_eda_e_baseline_prf2025_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto ADS033 — Aprendizagem de Máquina
## Classificação da gravidade de sinistros rodoviários federais — PRF 2025

**Integrantes:** Carlos Eduardo Mendonça, Ana Clara e Nathalya  
**Professor:** Rodrigo Gonçalves  
**Turma:** ADSDM2C — 2026/2  
**Marco 1:** auditoria, EDA, pré-processamento e baseline

> **Pergunta:** é possível classificar a gravidade de um sinistro já registrado usando informações contextuais que não revelem diretamente o desfecho clínico?

Cada bloco apresenta método, resultado esperado, interpretação e limitação. O objetivo não é apenas obter código executável, mas sustentar cada decisão na arguição.

## 1. Objetivo, alvo e custo do erro

- `alvo_grave = 1`: pelo menos uma morte ou uma pessoa ferida gravemente.
- `alvo_grave = 0`: nenhuma morte e nenhum ferido grave.

O **recall da classe grave** é prioritário porque mede quantos casos realmente graves foram identificados. Recall isolado não basta: prever tudo como grave produz recall de 100% e muitos falsos positivos. Por isso, também usamos precisão, F1, Average Precision e matriz de confusão.

**Momento da previsão:** classificamos um sinistro já registrado, não prevemos se um acidente ocorrerá. `pessoas` e `veiculos` entram na EDA, mas ficam fora do modelo contextual inicial, pois a disponibilidade dessas contagens no instante da triagem ainda precisa ser defendida.

## 2. Bibliotecas e reprodutibilidade

`random_state=42` permite repetir a divisão e comparar modelos nas mesmas condições. Não transforma o fenômeno em determinístico.

In [ ]:
# Importa recursos de sistema para validar o caminho da base.
import os
# Importa NumPy para vetorização e representação cíclica do tempo.
import numpy as np
# Importa pandas para leitura, auditoria e transformação tabular.
import pandas as pd
# Importa Plotly Express para gráficos interativos de alto nível.
import plotly.express as px
# Importa objetos gráficos para figuras específicas.
import plotly.graph_objects as go
# Importa o recurso de múltiplos painéis.
from plotly.subplots import make_subplots
# Importa a divisão estratificada de treino e teste.
from sklearn.model_selection import train_test_split
# Importa o transformador que aplica tratamentos por tipo de coluna.
from sklearn.compose import ColumnTransformer
# Importa codificação nominal e padronização numérica.
from sklearn.preprocessing import OneHotEncoder, StandardScaler
# Importa o pipeline que evita ajustes fora do treino.
from sklearn.pipeline import Pipeline
# Importa o classificador de referência.
from sklearn.dummy import DummyClassifier
# Importa as métricas do protocolo.
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, average_precision_score, confusion_matrix,
                             classification_report, precision_recall_curve)
# Centraliza a semente do experimento.
RANDOM_STATE = 42
# Centraliza o tema dos gráficos.
TEMA_PLOTLY = "plotly_white"
# Permite visualizar todas as colunas durante a auditoria.
pd.set_option("display.max_columns", None)
# Reduz quebras artificiais de tabelas na saída.
pd.set_option("display.width", 140)
print(f"Semente do experimento: {RANDOM_STATE}")

Semente do experimento: 42


## 3. Google Drive e leitura controlada

A validação deve falhar cedo se o arquivo não estiver no caminho oficial. Isso produz um erro compreensível antes que células dependentes sejam executadas.

In [ ]:
# Importa a montagem do Google Drive no Colab.
from google.colab import drive
# Monta o Drive sem forçar remontagem desnecessária.
drive.mount("/content/drive", force_remount=False)
# Define o caminho oficial acordado pelo squad.
CAMINHO_BASE = "/content/drive/MyDrive/base_compartilhada/datatran2025.csv"
# Interrompe a execução se o arquivo não existir.
if not os.path.exists(CAMINHO_BASE):
    raise FileNotFoundError(f"Arquivo não encontrado em: {CAMINHO_BASE}")
# Lê o CSV com separador, decimal e codificação da base.
df = pd.read_csv(CAMINHO_BASE, sep=";", decimal=",", encoding="latin1", low_memory=False)
# Exibe a dimensão recebida.
print(f"Base carregada: {df.shape[0]:,} linhas e {df.shape[1]} colunas".replace(",", "."))

Mounted at /content/drive
Base carregada: 72.529 linhas e 30 colunas


### 3.1 Contrato mínimo do dataset

Os testes abaixo verificam se o arquivo corresponde à versão consolidada esperada. Eles não provam que o conteúdo está semanticamente correto.

In [ ]:
# Converte a data; datas inválidas tornam-se NaT para auditoria.
df["data_inversa"] = pd.to_datetime(df["data_inversa"], errors="coerce")
# Converte o horário em duração desde meia-noite.
df["horario_convertido"] = pd.to_timedelta(df["horario"], errors="coerce")
# Reúne as colunas sem as quais o projeto não pode prosseguir.
COLUNAS_OBRIGATORIAS = {"id", "data_inversa", "horario", "uf", "br", "km",
    "classificacao_acidente", "fase_dia", "tipo_pista", "uso_solo", "pessoas",
    "mortos", "feridos_leves", "feridos_graves", "ilesos", "ignorados",
    "feridos", "veiculos"}
# Calcula se faltou alguma coluna estrutural.
colunas_ausentes = COLUNAS_OBRIGATORIAS.difference(df.columns)
# Valida a quantidade oficial de registros.
assert df.shape[0] == 72_529, "Quantidade de linhas inesperada."
# Valida as 30 colunas originais mais a conversão auxiliar do horário.
assert df.shape[1] == 31, "Quantidade de colunas inesperada após conversão."
# Valida a presença das colunas essenciais.
assert not colunas_ausentes, f"Colunas ausentes: {sorted(colunas_ausentes)}"
# Valida que o identificador está completo e é único.
assert df["id"].notna().all() and df["id"].is_unique, "Falha na chave id."
# Valida que data e horário foram convertidos sem perdas.
assert df["data_inversa"].notna().all(), "Existem datas inválidas."
assert df["horario_convertido"].notna().all(), "Existem horários inválidos."
# Valida a cobertura integral de 2025.
assert df["data_inversa"].min() == pd.Timestamp("2025-01-01")
assert df["data_inversa"].max() == pd.Timestamp("2025-12-31")
print("Contrato mínimo validado.")

Contrato mínimo validado.


## 4. Reconhecimento estrutural

Amostra, tipos e resumo numérico revelam leituras incorretas, escalas incompatíveis e categorias tratadas indevidamente como quantidades.

In [ ]:
# Exibe uma amostra para verificar formatos e valores.
display(df.head())
# Exibe tipos, não nulos e memória aproximada.
df.info()
# Resume as variáveis numéricas.
display(df.describe(include=[np.number]).T)
# Conta duplicatas integrais, ignorando apenas a coluna auxiliar derivada.
duplicatas_integrais = int(df.drop(columns="horario_convertido").duplicated().sum())
# Conta IDs repetidos.
duplicatas_id = int(df["id"].duplicated().sum())
# Organiza a auditoria de duplicidade.
resumo_duplicatas = pd.DataFrame({"verificacao": ["Duplicatas integrais", "IDs duplicados"],
                                  "quantidade": [duplicatas_integrais, duplicatas_id]})
display(resumo_duplicatas)

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,horario_convertido
0,652493,2025-01-01,quarta-feira,06:20:00,SP,116,225.0,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,-23.485868,-46.540753,SPRF-SP,DEL01-SP,UOP01-DEL01-SP,0 days 06:20:00
1,652519,2025-01-01,quarta-feira,07:50:00,CE,116,546.2,PENAFORTE,Pista esburacada,Colisão frontal,NaN,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,-7.812288,-39.083333,SPRF-CE,DEL05-CE,UOP03-DEL05-CE,0 days 07:50:00
2,652522,2025-01-01,quarta-feira,08:45:00,PR,369,88.2,CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,-23.182565,-50.637228,SPRF-PR,DEL07-PR,UOP05-DEL07-PR,0 days 08:45:00
3,652544,2025-01-01,quarta-feira,11:00:00,PR,116,74.0,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,-25.365177,-49.042230,SPRF-PR,DEL01-PR,UOP02-DEL01-PR,0 days 11:00:00
4,652549,2025-01-01,quarta-feira,09:30:00,MG,251,471.0,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,-16.468013,-43.431213,SPRF-MG,DEL12-MG,UOP01-DEL12-MG,0 days 09:30:00


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72529 entries, 0 to 72528
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype          
---  ------                  --------------  -----          
 0   id                      72529 non-null  int64          
 1   data_inversa            72529 non-null  datetime64[ns] 
 2   dia_semana              72529 non-null  object         
 3   horario                 72529 non-null  object         
 4   uf                      72529 non-null  object         
 5   br                      72529 non-null  int64          
 6   km                      72529 non-null  float64        
 7   municipio               72529 non-null  object         
 8   causa_acidente          72529 non-null  object         
 9   tipo_acidente           72529 non-null  object         
 10  classificacao_acidente  72528 non-null  object         
 11  fase_dia                72529 non-null  object         
 12  sentido_via             72529 no

,count,mean,std,min,25%,50%,75%,max
id,72529.0,700880.500862,26988.975438,652468.0,672804.0,703877.0,723739.0,758186.0
br,72529.0,208.502392,128.572019,0.0,101.0,155.0,319.0,495.0
km,72529.0,260.042406,227.921035,0.0,76.0,192.5,411.0,1257.0
pessoas,72529.0,2.596837,2.255139,1.0,2.0,2.0,3.0,76.0
mortos,72529.0,0.083318,0.338685,0.0,0.0,0.0,0.0,16.0
feridos_leves,72529.0,0.875953,1.037589,0.0,0.0,1.0,1.0,41.0
feridos_graves,72529.0,0.276,0.61119,0.0,0.0,0.0,0.0,22.0
ilesos,72529.0,1.053454,1.833533,0.0,0.0,1.0,1.0,71.0
ignorados,72529.0,0.394739,0.860584,0.0,0.0,0.0,1.0,81.0
feridos,72529.0,1.151953,1.143725,0.0,1.0,1.0,1.0,49.0


,verificacao,quantidade
0,Duplicatas integrais,0
1,IDs duplicados,0


**Interpretação:** a base possui 72.529 registros, sem duplicatas integrais ou IDs repetidos. Isso reduz risco de contagem dupla, mas não garante consistência entre os campos.

## 5. Completude: ausência física e desconhecimento semântico

`NaN` é ausência estrutural. `Ignorado` é uma categoria textual que registra desconhecimento e não aparece em `isna()`. Imputação automática não será aplicada.

In [ ]:
# Conta ausências físicas por coluna.
quantidade_ausentes = df.isna().sum()
# Calcula o percentual de ausências.
percentual_ausentes = quantidade_ausentes.div(len(df)).mul(100)
# Mantém apenas colunas com pelo menos uma ausência.
resumo_ausentes = pd.DataFrame({"ausentes": quantidade_ausentes,
    "percentual": percentual_ausentes}).query("ausentes > 0").sort_values("ausentes", ascending=False)
display(resumo_ausentes.round(4))
# Localiza a categoria exata Ignorado nas colunas textuais, sem percorrer linhas.
mascara_ignorado = df.select_dtypes(include="object").eq("Ignorado")
# Conta o desconhecimento semântico por coluna.
quantidade_ignorado = mascara_ignorado.sum().sort_values(ascending=False)
# Mantém somente colunas afetadas.
resumo_ignorado = quantidade_ignorado[quantidade_ignorado.gt(0)].to_frame("quantidade_ignorado")
# Calcula o percentual correspondente.
resumo_ignorado["percentual"] = resumo_ignorado["quantidade_ignorado"].div(len(df)).mul(100)
display(resumo_ignorado.round(4))

,ausentes,percentual
uop,38,0.0524
delegacia,22,0.0303
regional,2,0.0028
classificacao_acidente,1,0.0014


,quantidade_ignorado,percentual
condicao_metereologica,1000,1.3788


**Interpretação:** os `NaN` concentram-se em poucos campos administrativos e em um caso de `classificacao_acidente`. Categorias `Ignorado` expressam incerteza do registro, não condição observada. Nenhuma linha será excluída apenas por essas situações.

## 6. Consistência física e contábil

Testamos identidades do domínio por operações vetorizadas. Uma divergência indica necessidade de investigação, não prova a causa do problema.

In [ ]:
# Verifica se feridos é igual à soma de leves e graves.
df["feridos_consistente"] = df["feridos"].eq(df["feridos_leves"] + df["feridos_graves"])
# Soma as categorias de pessoas.
df["soma_categorias_pessoas"] = df["mortos"] + df["feridos"] + df["ilesos"] + df["ignorados"]
# Preserva sentido e magnitude da diferença.
df["diferenca_contagem_pessoas"] = df["pessoas"] - df["soma_categorias_pessoas"]
# Marca linhas cuja contabilidade fecha.
df["contagem_pessoas_consistente"] = df["diferenca_contagem_pessoas"].eq(0)
# Resume as quatro regras auditadas.
resumo_consistencia = pd.DataFrame({"regra": ["feridos = leves + graves",
    "pessoas = mortos + feridos + ilesos + ignorados", "pessoas >= 1", "veiculos >= 1"],
    "divergencias": [int((~df["feridos_consistente"]).sum()),
                     int((~df["contagem_pessoas_consistente"]).sum()),
                     int(df["pessoas"].lt(1).sum()), int(df["veiculos"].lt(1).sum())]})
# Calcula o percentual afetado.
resumo_consistencia["percentual"] = resumo_consistencia["divergencias"].div(len(df)).mul(100)
display(resumo_consistencia.round(3))

,regra,divergencias,percentual
0,feridos = leves + graves,0,0.000
1,pessoas = mortos + feridos + ilesos + ignorados,3823,5.271
2,pessoas >= 1,0,0.000
3,veiculos >= 1,0,0.000


In [ ]:
# Seleciona as linhas divergentes sem alterar a base original.
divergencias_pessoas = df.loc[~df["contagem_pessoas_consistente"]].copy()
# Conta a distribuição das diferenças.
distribuicao_diferencas = (divergencias_pessoas["diferenca_contagem_pessoas"]
    .value_counts().sort_index().rename_axis("diferenca").reset_index(name="quantidade"))
display(distribuicao_diferencas)
# Calcula a magnitude absoluta e seleciona os dez extremos.
casos_extremos = divergencias_pessoas.assign(
    diferenca_absoluta=divergencias_pessoas["diferenca_contagem_pessoas"].abs()
).nlargest(10, "diferenca_absoluta")
# Exibe os campos necessários à inspeção.
display(casos_extremos[["id", "uf", "pessoas", "mortos", "feridos", "ilesos",
                        "ignorados", "soma_categorias_pessoas", "diferenca_contagem_pessoas"]])

,diferenca,quantidade
0,-80,1
1,-14,1
2,-13,1
3,-11,11
4,-10,5
5,-9,1
6,-8,9
7,-7,11
8,-6,14
9,-5,61


,id,uf,pessoas,mortos,feridos,ilesos,ignorados,soma_categorias_pessoas,diferenca_contagem_pessoas
10260,707992,GO,2,0,0,1,81,82,-80
9474,704931,ES,11,1,8,0,16,25,-14
4945,685337,MT,2,0,0,1,14,15,-13
4193,670979,MT,8,1,2,4,12,19,-11
5926,689744,BA,2,0,0,1,12,13,-11
6789,693721,BA,2,0,0,1,12,13,-11
7469,696512,PR,3,0,1,1,12,14,-11
9043,703139,SP,2,0,0,1,12,13,-11
9585,705324,MG,4,0,1,2,12,15,-11
9805,706211,PE,2,0,0,1,12,13,-11


**Interpretação:** `feridos` fecha com suas componentes em toda a base. Em 3.823 registros, 5,27%, a soma das categorias supera `pessoas`. A maioria das diferenças é pequena, mas há extremos. A causa não pode ser inferida pelo teste; os registros serão preservados e submetidos a análise de sensibilidade.

## 7. Construção e validação do alvo

A regra usa somente as contagens necessárias para construir `y`. `classificacao_acidente` participa apenas da auditoria descritiva e ficará proibida em `X`.

In [ ]:
# Cria a classe 1 quando há morte ou ferido grave.
df["alvo_grave"] = (df["mortos"].gt(0) | df["feridos_graves"].gt(0)).astype("int8")
# Conta cada classe.
contagem_alvo = df["alvo_grave"].value_counts().sort_index()
# Calcula o percentual de cada classe.
percentual_alvo = df["alvo_grave"].value_counts(normalize=True).sort_index().mul(100)
# Organiza o resumo para exibição e gráfico.
resumo_alvo = pd.DataFrame({"classe": ["Não grave", "Grave"],
    "quantidade": contagem_alvo.reindex([0, 1]).to_numpy(),
    "percentual": percentual_alvo.reindex([0, 1]).to_numpy()})
display(resumo_alvo.round(2))
# Cruza a classificação oficial com o alvo derivado.
validacao_alvo = pd.crosstab(df["classificacao_acidente"].fillna("Ausente"),
                             df["alvo_grave"], margins=True)
display(validacao_alvo)

,classe,quantidade,percentual
0,Não grave,52036,71.75
1,Grave,20493,28.25


alvo_grave,0,1,All
classificacao_acidente,,,
Ausente,0,1,1
Com Vítimas Fatais,0,5209,5209
Com Vítimas Feridas,40898,15283,56181
Sem Vítimas,11138,0,11138
All,52036,20493,72529


**Interpretação:** fatais são graves e ocorrências sem vítimas são não graves. “Com Vítimas Feridas” inclui feridos leves e graves, por isso se divide entre as classes. Existe um caso grave sem classificação textual. A coerência do cruzamento não torna essa coluna elegível para modelagem.

## 8. Gráfico 1 — Distribuição do alvo

O gráfico evidencia o desbalanceamento que torna insuficiente avaliar somente a acurácia.

In [ ]:
# Cria rótulos com quantidade e percentual.
resumo_alvo["rotulo"] = (resumo_alvo["quantidade"].map(lambda v: f"{v:,.0f}".replace(",", "."))
    + " (" + resumo_alvo["percentual"].map(lambda v: f"{v:.2f}%".replace(".", ",")) + ")")
# Constrói o gráfico de barras.
fig_alvo = px.bar(resumo_alvo, x="classe", y="quantidade", text="rotulo", color="classe",
    color_discrete_map={"Não grave": "#4C78A8", "Grave": "#E45756"},
    title="Distribuição de alvo_grave — PRF 2025",
    labels={"classe": "Classe", "quantidade": "Ocorrências"}, template=TEMA_PLOTLY)
# Posiciona rótulos e remove legenda redundante.
fig_alvo.update_traces(textposition="outside")
fig_alvo.update_layout(showlegend=False)
fig_alvo.show()

**Leitura crítica:** 71,75% dos registros são não graves e 28,25% são graves. Um classificador pode parecer razoável pela acurácia e ainda falhar em todos os graves.

## 9. Posição, dispersão, cauda e zeros

Média e desvio-padrão serão acompanhados por mediana, quartis, IQR, extremos e percentual de zeros. `ddof=1` trata 2025 como uma realização de um processo rodoviário mais amplo.

In [ ]:
# Seleciona variáveis de contagem relevantes à EDA.
colunas_contagem = ["pessoas", "veiculos", "mortos", "feridos_leves",
                    "feridos_graves", "ilesos", "ignorados", "feridos"]
# Calcula medidas vetorizadas para todas as colunas.
resumo_descritivo = pd.DataFrame({"minimo": df[colunas_contagem].min(),
    "q1": df[colunas_contagem].quantile(0.25), "mediana": df[colunas_contagem].median(),
    "media": df[colunas_contagem].mean(), "q3": df[colunas_contagem].quantile(0.75),
    "maximo": df[colunas_contagem].max(), "desvio_padrao": df[colunas_contagem].std(ddof=1),
    "iqr": df[colunas_contagem].quantile(0.75) - df[colunas_contagem].quantile(0.25),
    "percentual_zeros": df[colunas_contagem].eq(0).mean().mul(100)})
# Calcula CV apenas quando a média não é zero.
resumo_descritivo["cv_percentual"] = (resumo_descritivo["desvio_padrao"]
    .div(resumo_descritivo["media"].replace(0, np.nan)).mul(100))
display(resumo_descritivo.round(3))

,minimo,q1,mediana,media,q3,maximo,desvio_padrao,iqr,percentual_zeros,cv_percentual
pessoas,1,2.0,2.0,2.597,3.0,76,2.255,1.0,0.000,86.842
veiculos,1,1.0,2.0,1.998,2.0,82,1.126,1.0,0.000,56.361
mortos,0,0.0,0.0,0.083,0.0,16,0.339,0.0,92.817,406.495
feridos_leves,0,0.0,1.0,0.876,1.0,41,1.038,1.0,36.624,118.453
feridos_graves,0,0.0,0.0,0.276,0.0,22,0.611,0.0,77.452,221.446
ilesos,0,0.0,1.0,1.053,1.0,71,1.834,1.0,37.683,174.050
ignorados,0,0.0,0.0,0.395,1.0,81,0.861,1.0,72.622,218.014
feridos,0,1.0,1.0,1.152,1.0,49,1.144,0.0,19.974,99.286


**Leitura crítica:** médias acima das medianas e máximos distantes de Q3 indicam caudas à direita. Em `mortos` e `feridos_graves`, o CV alto também reflete muitos zeros e média pequena; não deve ser interpretado sozinho.

## 10. Gráfico 2 — Pessoas: centro e cauda

O painel esquerdo vai até o percentil 99. O direito preserva a base completa e usa escala logarítmica na frequência. O recorte é apenas visual.

In [ ]:
# Calcula o percentil 99 e a quantidade acima dele.
p99_pessoas = int(df["pessoas"].quantile(0.99))
quantidade_acima_p99 = int(df["pessoas"].gt(p99_pessoas).sum())
# Cria dois painéis complementares.
fig_pessoas = make_subplots(rows=1, cols=2,
    subplot_titles=(f"Faixa principal até P99 = {p99_pessoas}", "Base completa em escala log"))
# Adiciona o histograma principal com caixas inteiras.
fig_pessoas.add_trace(go.Histogram(x=df.loc[df["pessoas"].le(p99_pessoas), "pessoas"],
    xbins={"start": 0.5, "end": p99_pessoas + 0.5, "size": 1}, marker_color="#4C78A8"), row=1, col=1)
# Adiciona o histograma completo.
fig_pessoas.add_trace(go.Histogram(x=df["pessoas"],
    xbins={"start": 0.5, "end": int(df["pessoas"].max()) + 0.5, "size": 1},
    marker_color="#F58518"), row=1, col=2)
# Revela frequências pequenas no segundo painel.
fig_pessoas.update_yaxes(type="log", row=1, col=2)
# Formata a figura.
fig_pessoas.update_layout(title="Distribuição de pessoas por ocorrência", template=TEMA_PLOTLY,
                          height=480, showlegend=False, bargap=0.05)
fig_pessoas.update_xaxes(title_text="Pessoas")
fig_pessoas.update_yaxes(title_text="Ocorrências", row=1, col=1)
fig_pessoas.update_yaxes(title_text="Ocorrências, escala log", row=1, col=2)
fig_pessoas.show()
print(f"Registros acima do P99, ocultos apenas no painel esquerdo: {quantidade_acima_p99}")

Registros acima do P99, ocultos apenas no painel esquerdo: 639


**Leitura crítica:** mediana 2, média próxima de 2,60 e máximo 76 indicam assimetria à direita. O gráfico não autoriza atribuir a cauda a ônibus ou colisões múltiplas sem cruzamentos adicionais.

## 11. Tukey e Gráfico 3 — Veículos

Tukey sinaliza observações para investigação. Não declara automaticamente erro e não autoriza exclusão.

In [ ]:
# Calcula quartis e IQR de veículos.
q1_veiculos = df["veiculos"].quantile(0.25)
q3_veiculos = df["veiculos"].quantile(0.75)
iqr_veiculos = q3_veiculos - q1_veiculos
# Calcula os limites da regra de Tukey.
limite_inferior_veiculos = q1_veiculos - 1.5 * iqr_veiculos
limite_superior_veiculos = q3_veiculos + 1.5 * iqr_veiculos
# Sinaliza vetorialmente valores fora dos limites.
df["veiculos_sinalizado_tukey"] = (df["veiculos"].lt(limite_inferior_veiculos)
    | df["veiculos"].gt(limite_superior_veiculos))
# Resume a aplicação matemática.
resumo_tukey = pd.DataFrame({"q1": [q1_veiculos], "q3": [q3_veiculos],
    "iqr": [iqr_veiculos], "limite_inferior": [limite_inferior_veiculos],
    "limite_superior": [limite_superior_veiculos],
    "sinalizados": [int(df["veiculos_sinalizado_tukey"].sum())],
    "percentual": [df["veiculos_sinalizado_tukey"].mean() * 100]})
display(resumo_tukey.round(3))
# Exibe todos os registros no boxplot.
fig_box = px.box(df, x="veiculos", points="outliers", template=TEMA_PLOTLY,
    title="Veículos por ocorrência e sinalização visual", labels={"veiculos": "Veículos"})
fig_box.show()
# Compara gravidade mantendo denominadores.
grave_por_tukey = (df.groupby("veiculos_sinalizado_tukey", as_index=False)
    .agg(ocorrencias=("id", "size"), percentual_grave=("alvo_grave", "mean")))
grave_por_tukey["percentual_grave"] = grave_por_tukey["percentual_grave"].mul(100)
display(grave_por_tukey.round(2))

,q1,q3,iqr,limite_inferior,limite_superior,sinalizados,percentual
0,1.0,2.0,1.0,-0.5,3.5,5219,7.196


,veiculos_sinalizado_tukey,ocorrencias,percentual_grave
0,False,67310,27.46
1,True,5219,38.46


**Leitura crítica:** quatro ou mais veículos são sinalizados, pois o limite superior é 3,5. O grupo sinalizado apresenta maior proporção de graves, mas isso é associação descritiva, não causalidade nem prova de ganho preditivo.

## 12. Gráfico 4 — Causas mais frequentes

Frequência responde quantas vezes cada causa aparece, não qual causa apresenta maior gravidade.

In [ ]:
# Conta as dez causas mais frequentes.
top_causas = (df["causa_acidente"].value_counts().head(10).sort_values()
    .rename_axis("causa_acidente").reset_index(name="ocorrencias"))
# Cria barras horizontais para rótulos longos.
fig_causas = px.bar(top_causas, x="ocorrencias", y="causa_acidente", orientation="h",
    text="ocorrencias", template=TEMA_PLOTLY, title="Dez causas mais frequentes",
    labels={"ocorrencias": "Ocorrências", "causa_acidente": "Causa registrada"})
fig_causas.update_traces(textposition="outside")
fig_causas.update_layout(height=580, margin={"l": 320})
fig_causas.show()

**Leitura crítica:** o gráfico descreve volume. Não é correto concluir que as causas mais frequentes são as mais graves ou que uma categoria textual prova um mecanismo causal.

## 13. Gráfico 5 — Gravidade por causa com suporte

Para evitar percentuais extremos baseados em pouquíssimos casos, mostramos apenas grupos com pelo menos 300 ocorrências. O filtro é visual e não remove categorias da base.

In [ ]:
# Resume suporte e proporção de graves por causa.
resumo_causa = (df.groupby("causa_acidente", as_index=False)
    .agg(ocorrencias=("id", "size"), percentual_grave=("alvo_grave", "mean")))
# Mantém grupos com suporte mínimo e converte a taxa em percentual.
causas_com_suporte = resumo_causa.loc[resumo_causa["ocorrencias"].ge(300)].copy()
causas_com_suporte["percentual_grave"] = causas_com_suporte["percentual_grave"].mul(100)
# Seleciona as quinze maiores taxas.
top_causas_graves = causas_com_suporte.nlargest(15, "percentual_grave").sort_values("percentual_grave")
# Cria rótulos que preservam o denominador.
top_causas_graves["rotulo"] = (top_causas_graves["percentual_grave"].map(lambda v: f"{v:.1f}%")
    + " | n=" + top_causas_graves["ocorrencias"].astype(str))
# Cria o gráfico horizontal.
fig_causas_graves = px.bar(top_causas_graves, x="percentual_grave", y="causa_acidente",
    orientation="h", text="rotulo", color="percentual_grave", color_continuous_scale="Reds",
    template=TEMA_PLOTLY, title="Maiores proporções de graves por causa, n ≥ 300",
    labels={"percentual_grave": "Casos graves (%)", "causa_acidente": "Causa registrada"})
fig_causas_graves.update_traces(textposition="outside")
fig_causas_graves.update_layout(height=650, margin={"l": 330}, coloraxis_showscale=False)
fig_causas_graves.show()

**Leitura crítica:** taxa e frequência são diferentes. Mesmo com suporte mínimo, composição geográfica, pista, horário e tipo de acidente podem confundir as associações; o gráfico não estabelece causalidade.

## 14. Gráfico 6 — Gravidade por tipo de pista

A taxa é calculada dentro de cada tipo de pista e acompanhada do denominador.

In [ ]:
# Resume quantidade e proporção de graves por tipo de pista.
resumo_pista = (df.groupby("tipo_pista", as_index=False)
    .agg(ocorrencias=("id", "size"), percentual_grave=("alvo_grave", "mean")))
# Converte para percentual.
resumo_pista["percentual_grave"] = resumo_pista["percentual_grave"].mul(100)
# Cria rótulos com taxa e denominador.
resumo_pista["rotulo"] = (resumo_pista["percentual_grave"].map(lambda v: f"{v:.2f}%")
    + " | n=" + resumo_pista["ocorrencias"].map(lambda v: f"{v:,}".replace(",", ".")))
# Ordena e plota.
resumo_pista = resumo_pista.sort_values("percentual_grave", ascending=False)
fig_pista = px.bar(resumo_pista, x="tipo_pista", y="percentual_grave", text="rotulo",
    color="tipo_pista", template=TEMA_PLOTLY, title="Casos graves por tipo de pista",
    labels={"tipo_pista": "Tipo de pista", "percentual_grave": "Casos graves (%)"})
fig_pista.update_traces(textposition="outside")
fig_pista.update_layout(showlegend=False)
fig_pista.show()

**Leitura crítica:** pista simples apresenta maior proporção observada. O código não demonstrou que colisões frontais ou ausência de barreira expliquem a diferença. A conclusão correta é associativa.

## 15. Subpopulações e risco de agregação enganosa

Comparamos a correlação `pessoas × veiculos` no total e por `uso_solo`. Mudança de intensidade indica heterogeneidade; não prova automaticamente Paradoxo de Simpson.

In [ ]:
# Calcula a correlação agregada.
correlacao_geral = df[["pessoas", "veiculos"]].corr().loc["pessoas", "veiculos"]
# Calcula a correlação dentro de cada grupo por uma agregação explícita.
correlacao_grupos = (df.groupby("uso_solo")[["pessoas", "veiculos"]]
    .corr().loc[(slice(None), "pessoas"), "veiculos"].droplevel(1)
    .rename("correlacao").reset_index())
# Acrescenta o valor agregado.
resumo_correlacoes = pd.concat([pd.DataFrame({"uso_solo": ["Base agregada"],
    "correlacao": [correlacao_geral]}), correlacao_grupos], ignore_index=True)
display(resumo_correlacoes.round(3))
# Plota a comparação.
fig_subgrupos = px.bar(resumo_correlacoes, x="uso_solo", y="correlacao",
    text=resumo_correlacoes["correlacao"].map(lambda v: f"{v:.3f}"), color="uso_solo",
    template=TEMA_PLOTLY, title="Correlação pessoas × veículos: total e por uso do solo",
    labels={"uso_solo": "Subpopulação", "correlacao": "Correlação de Pearson"})
fig_subgrupos.update_traces(textposition="outside")
fig_subgrupos.update_layout(showlegend=False)
fig_subgrupos.show()

,uso_solo,correlacao
0,Base agregada,0.395
1,Não,0.348
2,Sim,0.571


**Leitura crítica:** a intensidade varia entre subpopulações. Não há inversão demonstrada; portanto, o resultado ensina a investigar agregações, mas não autoriza declarar Simpson como fato consumado.

## 16. Correlação estrutural, somente para EDA

As variáveis de vítimas participam apenas da compreensão contábil. Correlações com componentes de `pessoas` podem ser mecânicas, e o alvo não entra porque foi construído com `mortos` e `feridos_graves`.

In [ ]:
# Seleciona contagens para auditoria estrutural, não para modelagem.
colunas_correlacao = ["pessoas", "veiculos", "mortos", "feridos_graves",
                      "feridos_leves", "ilesos", "ignorados"]
# Calcula Pearson entre as contagens.
matriz_correlacao = df[colunas_correlacao].corr(method="pearson")
# Cria o mapa de calor completo.
fig_correlacao = px.imshow(matriz_correlacao, text_auto=".2f", aspect="auto",
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
    title="Correlação estrutural entre contagens — uso exclusivo na EDA")
fig_correlacao.update_layout(template=TEMA_PLOTLY, height=650)
fig_correlacao.show()

**Leitura crítica:** uma correlação alta entre `pessoas` e `ilesos` não é descoberta causal independente, pois as variáveis pertencem à mesma contabilidade. O mapa identifica estrutura e redundância, não elegibilidade para `X`.

## 17. Atributos contextuais e proteção contra leakage

Proibidos: `mortos`, `feridos_graves`, `feridos_leves`, `feridos`, `ilesos`, `ignorados`, `classificacao_acidente`, `alvo_grave` e `id`. Nesta versão conservadora, também ficam fora `pessoas`, `veiculos`, causas, tipo do acidente, unidades administrativas, coordenadas, `km` isolado e `tracado_via`.

In [ ]:
# Deriva o mês da data.
df["mes"] = df["data_inversa"].dt.month.astype("int8")
# Converte horário em hora decimal.
componentes_horario = df["horario_convertido"].dt.components
df["hora_decimal"] = componentes_horario["hours"] + componentes_horario["minutes"].div(60) + componentes_horario["seconds"].div(3600)
# Representa mês e hora como ciclos, aproximando dezembro de janeiro e 23h59 de 0h.
df["mes_seno"] = np.sin(2 * np.pi * df["mes"] / 12)
df["mes_cosseno"] = np.cos(2 * np.pi * df["mes"] / 12)
df["hora_seno"] = np.sin(2 * np.pi * df["hora_decimal"] / 24)
df["hora_cosseno"] = np.cos(2 * np.pi * df["hora_decimal"] / 24)
# Define atributos numéricos contextuais.
COLUNAS_NUMERICAS = ["mes_seno", "mes_cosseno", "hora_seno", "hora_cosseno"]
# Define atributos categóricos nominais.
COLUNAS_CATEGORICAS = ["dia_semana", "uf", "br", "fase_dia", "sentido_via",
                        "condicao_metereologica", "tipo_pista", "uso_solo"]
# Separa X e y.
X = df[COLUNAS_NUMERICAS + COLUNAS_CATEGORICAS].copy()
y = df["alvo_grave"].copy()
# Define a lista de segurança contra vazamento.
COLUNAS_PROIBIDAS = {"id", "mortos", "feridos_graves", "feridos_leves", "feridos",
    "ilesos", "ignorados", "classificacao_acidente", "alvo_grave"}
# Interrompe se alguma proibida entrar em X.
intersecao_proibida = COLUNAS_PROIBIDAS.intersection(X.columns)
assert not intersecao_proibida, f"Vazamento em X: {sorted(intersecao_proibida)}"
print("Dimensão de X:", X.shape)
print("Numéricas:", COLUNAS_NUMERICAS)
print("Categóricas:", COLUNAS_CATEGORICAS)

Dimensão de X: (72529, 12)
Numéricas: ['mes_seno', 'mes_cosseno', 'hora_seno', 'hora_cosseno']
Categóricas: ['dia_semana', 'uf', 'br', 'fase_dia', 'sentido_via', 'condicao_metereologica', 'tipo_pista', 'uso_solo']


## 18. Split 70/30 estratificado

A divisão ocorre antes de qualquer `fit`. O teste simula dados não vistos e não deve orientar hiperparâmetros, balanceamento ou limiar.

In [ ]:
# Divide a base preservando aproximadamente a proporção das classes.
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.30,
    stratify=y, random_state=RANDOM_STATE)
# Resume tamanho e graves por partição.
resumo_split = pd.DataFrame({"particao": ["Treino", "Teste"],
    "registros": [len(y_treino), len(y_teste)], "graves": [int(y_treino.sum()), int(y_teste.sum())],
    "percentual_graves": [y_treino.mean() * 100, y_teste.mean() * 100]})
display(resumo_split.round(3))
# Valida independência dos índices e preservação das linhas.
assert X_treino.index.intersection(X_teste.index).empty
assert len(X_treino) + len(X_teste) == len(X)
print("Split validado.")

,particao,registros,graves,percentual_graves
0,Treino,50770,14345,28.255
1,Teste,21759,6148,28.255


Split validado.


## 19. Pré-processamento sem vazamento

O `ColumnTransformer` aplica tratamentos por tipo. O `Pipeline` confina o ajuste ao treino. Não há imputação porque os atributos selecionados não possuem `NaN`; novas colunas exigirão imputadores dentro do pipeline.

In [ ]:
# Cria a padronização das variáveis numéricas.
preprocessador_numerico = Pipeline(steps=[("padronizacao", StandardScaler())])
# Cria o one-hot nominal, tolerando categorias futuras.
preprocessador_categorico = Pipeline(steps=[("one_hot", OneHotEncoder(handle_unknown="ignore"))])
# Encaminha cada grupo ao tratamento correto.
preprocessador = ColumnTransformer(transformers=[
    ("numericas", preprocessador_numerico, COLUNAS_NUMERICAS),
    ("categoricas", preprocessador_categorico, COLUNAS_CATEGORICAS)], remainder="drop")
# Cria um pipeline para auditar a transformação.
pipeline_preprocessamento = Pipeline(steps=[("preprocessamento", preprocessador)])
# Ajusta parâmetros somente no treino.
X_treino_transformado = pipeline_preprocessamento.fit_transform(X_treino)
# Apenas transforma o teste.
X_teste_transformado = pipeline_preprocessamento.transform(X_teste)
# Recupera os nomes gerados.
nomes_transformados = pipeline_preprocessamento.named_steps["preprocessamento"].get_feature_names_out()
print("Treino antes:", X_treino.shape)
print("Treino depois:", X_treino_transformado.shape)
print("Teste depois:", X_teste_transformado.shape)
# Valida preservação das linhas.
assert X_treino_transformado.shape[0] == X_treino.shape[0]
assert X_teste_transformado.shape[0] == X_teste.shape[0]
display(pd.Series(nomes_transformados, name="atributo_transformado").head(25))

Treino antes: (50770, 12)
Treino depois: (50770, 174)
Teste depois: (21759, 174)


,atributo_transformado
0,numericas__mes_seno
1,numericas__mes_cosseno
2,numericas__hora_seno
3,numericas__hora_cosseno
4,categoricas__dia_semana_domingo
5,categoricas__dia_semana_quarta-feira
6,categoricas__dia_semana_quinta-feira
7,categoricas__dia_semana_segunda-feira
8,categoricas__dia_semana_sexta-feira
9,categoricas__dia_semana_sábado


## 20. Baseline majoritário

O `DummyClassifier` aprende somente a classe mais frequente. Ele não aprende relações entre atributos e alvo. `random_state=42` é mantido por padronização, embora esta estratégia seja determinística.

In [ ]:
# Cria o modelo de referência.
modelo_baseline = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
# Encadeia pré-processamento e modelo.
pipeline_baseline = Pipeline(steps=[("preprocessamento", preprocessador),
                                    ("modelo", modelo_baseline)])
# Ajusta somente no treino.
pipeline_baseline.fit(X_treino, y_treino)
# Prediz classes no teste.
y_pred_baseline = pipeline_baseline.predict(X_teste)
# Obtém o escore da classe grave para Average Precision.
y_score_baseline = pipeline_baseline.predict_proba(X_teste)[:, 1]
print("Classes previstas:", np.unique(y_pred_baseline))

Classes previstas: [0]


## 21. Métricas do baseline

`zero_division=0` registra precisão e F1 como zero quando nenhuma ocorrência é prevista como grave.

In [ ]:
# Calcula acurácia global.
acuracia = accuracy_score(y_teste, y_pred_baseline)
# Calcula métricas da classe grave.
precisao = precision_score(y_teste, y_pred_baseline, pos_label=1, zero_division=0)
recall = recall_score(y_teste, y_pred_baseline, pos_label=1, zero_division=0)
f1 = f1_score(y_teste, y_pred_baseline, pos_label=1, zero_division=0)
# Calcula Average Precision com escores, não classes finais.
average_precision = average_precision_score(y_teste, y_score_baseline)
# Organiza e exibe em percentual.
metricas = pd.DataFrame({"metrica": ["Acurácia", "Precisão grave", "Recall grave", "F1 grave", "Average Precision"],
                         "percentual": np.array([acuracia, precisao, recall, f1, average_precision]) * 100})
display(metricas.round(2))
# Exibe o relatório por classe.
print(classification_report(y_teste, y_pred_baseline, digits=4, zero_division=0))

,metrica,percentual
0,Acurácia,71.75
1,Precisão grave,0.00
2,Recall grave,0.00
3,F1 grave,0.00
4,Average Precision,28.25


              precision    recall  f1-score   support

           0     0.7175    1.0000    0.8355     15611
           1     0.0000    0.0000    0.0000      6148

    accuracy                         0.7175     21759
   macro avg     0.3587    0.5000    0.4177     21759
weighted avg     0.5147    0.7175    0.5994     21759



## 22. Gráfico 7 — Matriz de confusão

Linhas representam classes reais; colunas, classes previstas.

In [ ]:
# Calcula a matriz com ordem fixa 0, 1.
matriz_confusao = confusion_matrix(y_teste, y_pred_baseline, labels=[0, 1])
# Cria o mapa de calor.
fig_matriz = px.imshow(matriz_confusao, x=["Não grave", "Grave"], y=["Não grave", "Grave"],
    text_auto=True, color_continuous_scale="Blues", template=TEMA_PLOTLY,
    title="Matriz de confusão — Dummy majoritário",
    labels={"x": "Classe prevista", "y": "Classe real", "color": "Ocorrências"})
fig_matriz.update_layout(height=520)
fig_matriz.show()

**Interpretação central:** o baseline alcança cerca de 71,75% de acurácia porque prevê tudo como não grave. Todos os 6.148 graves do teste são falsos negativos; recall, precisão e F1 da classe grave são zero. Average Precision fica próxima da prevalência positiva, pois não há ordenação útil de risco.

## 23. Curva precisão-recall

Com escore constante, o baseline não separa os casos. A prevalência da classe grave é a referência natural.

In [ ]:
# Calcula os pontos da curva PR.
precisoes, recalls, limiares = precision_recall_curve(y_teste, y_score_baseline)
# Calcula a prevalência positiva do teste.
prevalencia_teste = y_teste.mean()
# Cria a figura e adiciona a curva.
fig_pr = go.Figure()
fig_pr.add_trace(go.Scatter(x=recalls, y=precisoes, mode="lines+markers",
                            name="Dummy majoritário", line={"color": "#E45756", "width": 3}))
# Adiciona a referência da prevalência.
fig_pr.add_hline(y=prevalencia_teste, line_dash="dash", line_color="#4C78A8",
                 annotation_text=f"Prevalência = {prevalencia_teste:.2%}")
# Formata e exibe.
fig_pr.update_layout(title=f"Curva Precisão–Recall | AP = {average_precision:.3f}",
    xaxis_title="Recall grave", yaxis_title="Precisão grave", xaxis={"range": [0, 1.02]},
    yaxis={"range": [0, 1.02]}, template=TEMA_PLOTLY, height=520)
fig_pr.show()

## 24. Limitações

1. A base é observacional e não autoriza causalidade.
2. O alvo binário reúne ferimentos graves e mortes.
3. As 3.823 divergências exigem sensibilidade futura.
4. `pessoas` e `veiculos` foram excluídas do modelo contextual por dúvida temporal.
5. `tracado_via` exige tratamento multirrótulo.
6. Categorias raras devem ser tratadas dentro do treino.
7. O teste avaliou o baseline; decisões futuras devem ocorrer no treino por validação cruzada.
8. Desempenho de 2025 não garante generalização temporal.

## 25. Próximos passos

- **Nathalya:** k-means, cotovelo, silhueta, caracterização e Árvore de Decisão.
- **Carlos Eduardo:** k-NN, Naive Bayes, MLP, Stratified k-Fold e balanceamento dentro das dobras.
- **Squad:** comparar modelos no mesmo teste congelado por recall grave, precisão, F1, PR-AUC e matriz de confusão.
- **Revisão:** reiniciar a sessão do Colab, executar tudo e revisar o PR.

## 26. Versionamento Git

**Branch:** `feat/marco1-eda-baseline`  
**Responsável:** Ana Clara  
**Revisão:** Carlos Eduardo Mendonça

**Commit:**
```text
feat(eda): consolida auditoria, preprocessing e baseline do Marco 1
```

**PR:**
> Consolida o Marco 1 da base PRF 2025 com contrato do dataset, ausências físicas e semânticas, consistência, criação e validação do alvo, estatística descritiva, gráficos interpretados, prevenção de leakage, split estratificado, pré-processamento em pipeline e baseline avaliado por matriz de confusão, acurácia, precisão, recall, F1 e Average Precision.